# Imports

Recommended to run with /ix/rbao/images/pathml_jupyter6_nv.sif singularity container in order to have cv2 and pathml

In [49]:
import os
import sys
import cv2 #For blur detection
import json
import math
import time
import datetime
from tqdm import tqdm
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
from pathml.core import HESlide 
from pathml.preprocessing import TissueDetectionHE, LabelWhiteSpaceHE, LabelArtifactTileHE, StainNormalizationHE
from matplotlib import pyplot as plt
import matplotlib as mpl
prj = "HCC-CBS-210-RBao-HEpredict"
base = Path('/ix/rbao/Projects/%s' % prj)
data = base.joinpath('data','20240802_HN18139_HE_Jon','18-139_101-155')
helper_path = base.joinpath('scripts','pancaner_pipe','helpers')
sys.path.append(helper_path)
from helpers import anno as annoHelper
%load_ext autoreload
%autoreload 2
print(data.exists())
results = base.joinpath('results','hn','v1','tiles')
problem_file = 'he060.svs'
df = pd.read_csv(results.joinpath('%s_tiles_df.tsv' % problem_file.split('.')[0]),
                 sep='\t')
fn = data.joinpath(problem_file)
print(fn.exists(),df.shape)
df.head()

True
True (2008, 7)


,tile,x,y,sz,anno,anno_num,overlap
0,he060_n891_x8736_y2688_px224.jpg,8736.0,2688.0,224.0,notAnnotated,-1.0,0.0
1,he060_n892_x8960_y2688_px224.jpg,8960.0,2688.0,224.0,notAnnotated,-1.0,0.0
2,he060_n893_x9184_y2688_px224.jpg,9184.0,2688.0,224.0,notAnnotated,-1.0,0.0
3,he060_n894_x9408_y2688_px224.jpg,9408.0,2688.0,224.0,notAnnotated,-1.0,0.0
4,he060_n895_x9632_y2688_px224.jpg,9632.0,2688.0,224.0,notAnnotated,-1.0,0.0


In [16]:
wsi = HESlide(str(fn))
tissue_detect = TissueDetectionHE(mask_name='tissue',
                                  outer_contours_only= True,
                                  blur_ksize=21,
                                  threshold = 20) #Thresh possibly off for smaller tile size (<=224 )?

blank_detect=LabelWhiteSpaceHE(label_name='blank',
                               proportion_threshold=0.9, #Thresh too low?
                               )

art_detect = LabelArtifactTileHE(label_name='artifact')
normalizer = StainNormalizationHE(target = "normalize",
                                  stain_estimation_method = "macenko")


# TILE WSI AND FILTER TILES BASED ON VARIOUS CRITERIA:
tile_size = 224
tile_stride = tile_size # tile_size//2

tiles = wsi.generate_tiles(shape=tile_size,
                           stride=tile_stride,
                           pad=False,
                           level=0)

In [5]:
def collect_issues(skips_df,tile,i,iii,problem,value):
    y,x = tile.coords #Flipped to match qupath x,y in v9
    skips_df.loc[iii,'n'] = i
    skips_df.loc[iii,'problem']=problem
    skips_df.loc[iii,'value'] = value
    skips_df.loc[iii,'x'] = x
    skips_df.loc[iii,'y'] = y
    # im = np.array(tile.image).reshape(-1)
    # start = 5
    # stop = len(im) + start
    # skips_df.iloc[iii,start:stop] = im
    iii = iii + 1
    return skips_df, iii

In [17]:
n = df.tile.str.split('_').str[1].str.split('n').str[1].astype(int)
ii=0
tile_df = pd.DataFrame([])
skips_df = pd.DataFrame([])
iii = 0
bgr = (2,1,0)
tiles = wsi.generate_tiles(shape=tile_size,
                           stride=tile_stride,
                           pad=False,
                           level=0)
tile_size = 224
slide_num = 'he060'
tile_stride = tile_size # tile_size//2
tissue_thresh = 70 # % of tile that contains tissue
gj_anno_overlap = 10 # If assigning tiles to geojson this percent of tile must be inside annotation to be included
blur_thresh = 40 #threshold for blurriness calculated by strength of edges in image (higher = sharper image)
use_stain_norm = True
tot = 127 * tile_size**2
tot_tiles_est=(wsi.shape[0]//tile_size) * (wsi.shape[1]//tile_size) * (2*(tile_size//tile_stride))
t_stack=np.array([])
for i,tile in enumerate(tqdm(tiles)):
    if (not np.any(n == i)): #If tile was never saved in first pass, examine it again
        blank_detect.apply(tile)    
        if tile.labels['blank'] == False:
            art_detect.apply(tile)
            #In addition -- detect pen / glass edge artifacts??
            if tile.labels['artifact'] == False:           
                im = np.array(tile.image) # r g b page order
                blur_detect = cv2.Laplacian(im[:,:,bgr], cv2.CV_64F).var()
                if (blur_detect > blur_thresh):
                    tissue_detect.apply(tile)
                    tissue_mask=tile.masks['tissue']
                    per = 100 * np.sum(tissue_mask.flatten())/tot
                    if (per > tissue_thresh) :
                        if use_stain_norm:
                            normalizer.apply(tile)
                            im = np.array(tile.image) # r g b page order
                        y,x = tile.coords #Flipped to match qupath x,y in v9
                        tile_fn = '%s_n%d_x%d_y%d_px%d.jpg' % (slide_num,i,x,y,tile_size)
                        tile_df.loc[ii,'tile'] = tile_fn
                        tile_df.loc[ii,'x'] = x
                        tile_df.loc[ii,'y'] = y
                        tile_df.loc[ii,'sz'] = tile_size
                        ii+=1 #Counter of included tiles (excludes blanks/artifacts/blur etc)
                    else:
                        skips_df, iii = collect_issues(skips_df,tile, i,iii, 'percent_tissue',per)
                else:
                    skips_df, iii = collect_issues(skips_df,tile, i, iii, 'blur', blur_detect)
            else:
                skips_df, iii = collect_issues(skips_df,tile, i, iii, 'artifact',tile.labels['artifact'])
        else:
            skips_df, iii = collect_issues(skips_df,tile, i, iii, 'blank',tile.labels['blank'])


8662it [00:50, 171.69it/s]


In [11]:
newfn= results.parent.joinpath('%s_skipped_tiles_list.tsv' % slide_num)
skips_df.to_csv(newfn,sep = '\t')

In [10]:
results.parent

PosixPath('/ix/rbao/Projects/HCC-CBS-210-RBao-HEpredict/results/hn/v1')

In [12]:
skips_df.groupby('problem')['n'].count()

problem
artifact           517
blank             6006
percent_tissue     133
Name: n, dtype: int64

In [14]:
tile_df

,tile,x,y,sz
0,he060_n891_x8736_y2688_px224.jpg,8736.0,2688.0,224.0
1,he060_n892_x8960_y2688_px224.jpg,8960.0,2688.0,224.0
2,he060_n893_x9184_y2688_px224.jpg,9184.0,2688.0,224.0
3,he060_n894_x9408_y2688_px224.jpg,9408.0,2688.0,224.0
4,he060_n895_x9632_y2688_px224.jpg,9632.0,2688.0,224.0
...,...,...,...,...
1882,he060_n7916_x7840_y24864_px224.jpg,7840.0,24864.0,224.0
1883,he060_n7917_x8064_y24864_px224.jpg,8064.0,24864.0,224.0
1884,he060_n7918_x8288_y24864_px224.jpg,8288.0,24864.0,224.0
1885,he060_n7987_x7840_y25088_px224.jpg,7840.0,25088.0,224.0


# Examine regions known to have tumor tissue but tiles not being labeled as such:

In [57]:
x = 6368
y = 16082
# x = 4470
# y = 13154
idx = (df.x.values > x) & (df.x.values < (x + 500)) & \
      (df.y.values > y) & (df.y.values < (y + 500))
df.loc[idx,:]


,tile,x,y,sz,anno,anno_num,overlap
1099,he060_n5141_x6496_y16128_px224.jpg,6496.0,16128.0,224.0,notAnnotated,-1.0,0.0
1100,he060_n5142_x6720_y16128_px224.jpg,6720.0,16128.0,224.0,notAnnotated,-1.0,0.0
1132,he060_n5212_x6496_y16352_px224.jpg,6496.0,16352.0,224.0,notAnnotated,-1.0,0.0
1133,he060_n5213_x6720_y16352_px224.jpg,6720.0,16352.0,224.0,notAnnotated,-1.0,0.0
1165,he060_n5283_x6496_y16576_px224.jpg,6496.0,16576.0,224.0,notAnnotated,-1.0,0.0
1166,he060_n5284_x6720_y16576_px224.jpg,6720.0,16576.0,224.0,notAnnotated,-1.0,0.0


In [25]:
gj_fn =  data.parent.joinpath('18-139_101-155-qupath','18139_pt101to155','geojson','he060.geojson')
gj_fn.exists()

True

In [26]:
with open(gj_fn) as f:
            allobjects = json.load(f) #  

In [40]:
 coords = feat['geometry']['coordinates']
nearby=False
x = 6368
y = 16082
tile_xy = [x,y]
thresh = math.sqrt((tile_size**2)*2) * 2
if feat['geometry']['type'] != 'MultiPolygon':
    coords = [coords]
for i,multi_polygon in enumerate(coords):
    for ii, polygon in enumerate(multi_polygon):
        if len(polygon)>2:
            dat = np.array(polygon)
            min_x = np.min(np.sqrt((tile_xy[0] - dat[0,:])**2))
            min_y = np.min(np.sqrt((tile_xy[1] - dat[1,:])**2))
            if (min_x < thresh) | (min_y < thresh):
                nearby=True
                break;     
print(nearby)

False


In [45]:
thresh

633.5676759431466

In [44]:
min_x

1765.0

In [53]:
coords = feat['geometry']['coordinates']

if feat['geometry']['type'] != 'MultiPolygon':
    coords = [coords]
all_out = []
# x = 6368
# y = 16082
x = 4470
y = 13154
points = [[x,y]]
for ii,multi_polygon in enumerate(coords):
    for i, polygon in enumerate(multi_polygon):
        # out = np.zeros((len(points),1))
        path = mpl.path.Path(polygon)
        inside2 = path.contains_points(points)
        print(inside2)
        all_out.append(inside2)

[False]


In [65]:
# x = 6368
# y = 16082
# x = 4470
# y = 13154
x = 5900
y = 16793
idx = (df.x.values > x) & (df.x.values < (x + 500)) & \
      (df.y.values > y) & (df.y.values < (y + 500))
df.loc[idx,:]
max_overlap = 0
per_overlap = 0
# or enumerate(allobjects['features']) ? how to standardize?
subset = df.loc[idx,:].reset_index()
for i in range(0,subset.shape[0]):
    x = subset.loc[i,'x']
    y = subset.loc[i,'y']
    print(subset.loc[i,'tile'])
    tile_size = 224
    if isinstance(allobjects,list): #list of objects, possibly multipolygons
        enum_obj = allobjects
    elif 'features' in allobjects.keys(): #flat list of annotations
        enum_obj = allobjects['features']
    for iii,feat in enumerate(enum_obj): #Be aware that this structure can change
        if 'classification' in feat['properties'].keys():
            anno_type = feat['properties']['classification']['name'] 
        else:
            anno_type = 'anno'                                    
        print('Finding tiles in %s' % anno_type)
        per_overlap = annoHelper.check_tile_overlap_feat(feat,
                                                         [int(x),int(y)],
                                                         int(tile_size))
        nearby = annoHelper.check_tile_near_feature(feat,[x,y],tile_size)
        print(nearby)
        print('%d overlap found' % per_overlap)

he060_n5352_x6048_y16800_px224.jpg
Finding tiles in Tumor
True
88 overlap found
he060_n5353_x6272_y16800_px224.jpg
Finding tiles in Tumor
True
100 overlap found
he060_n5423_x6048_y17024_px224.jpg
Finding tiles in Tumor
True
92 overlap found
he060_n5424_x6272_y17024_px224.jpg
Finding tiles in Tumor
True
100 overlap found
he060_n5494_x6048_y17248_px224.jpg
Finding tiles in Tumor
True
66 overlap found
he060_n5495_x6272_y17248_px224.jpg
Finding tiles in Tumor
True
75 overlap found


In [32]:
n_p

0